# Quantum Linear Algebra

This notebook provides an in-depth exploration of the linear algebra concepts that form the mathematical foundation of quantum computing. Understanding these concepts is essential for grasping how quantum algorithms work and why they can provide computational advantages.

## Learning Objectives

By the end of this notebook, you will understand:

1. **Vector Spaces in Quantum Mechanics** - The Hilbert space formalism
2. **Matrix Operations** - Unitary, Hermitian, and normal matrices
3. **Eigenvalues and Eigenvectors** - Spectral decomposition
4. **Tensor Products** - Composing multi-qubit systems
5. **Linear Transformations** - Operators and their properties
6. **Singular Value Decomposition** - Classical and quantum perspectives

## Prerequisites

- Basic linear algebra (vectors, matrices, operations)
- Complex numbers fundamentals
- Python and NumPy proficiency
- Completion of quantum_computing_basics.ipynb recommended

---

## 1. Introduction: Why Linear Algebra for Quantum Computing?

Quantum mechanics is fundamentally a linear theory. The state of a quantum system is described by a vector in a complex vector space (Hilbert space), and the evolution of that state is described by linear transformations (operators).

This mathematical structure is what makes quantum computing possible:

- **Superposition** corresponds to linear combinations of basis states
- **Entanglement** arises from tensor products of Hilbert spaces
- **Quantum gates** are unitary matrices acting on state vectors
- **Measurement** involves projection operators

Let's build up these concepts systematically.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, Math, Latex

# Import Qiskit components
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_vector

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

print("✓ Environment ready for quantum linear algebra!")

## 2. Vector Spaces in Quantum Mechanics

### 2.1 The Complex Vector Space ℂ²

The state of a single qubit lives in a 2-dimensional complex vector space ℂ². This space has some special properties that distinguish it from ordinary real vector spaces.

A vector in this space can be written as:

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

where α and β are complex numbers. The **bra-ket** notation (Dirac notation) is standard in quantum mechanics:

- **Ket**: |ψ⟩ represents a column vector (a "state vector")
- **Bra**: ⟨ψ| represents the conjugate transpose (a row vector)
- **Bracket**: ⟨φ|ψ⟩ represents the inner product

In [ ]:
# Explore the structure of ℂ²

# Define basis states (computational basis)
ket_0 = np.array([1, 0], dtype=complex)  # |0⟩
ket_1 = np.array([0, 1], dtype=complex)  # |1⟩

print("=== Computational Basis States ===")
print(f"|0⟩ = {ket_0}")
print(f"|1⟩ = {ket_1}")

# Inner product (bra-ket)
print("\n=== Inner Products ===")
print(f"⟨0|0⟩ = {np.vdot(ket_0, ket_0)}")  # Should be 1
print(f"⟨1|1⟩ = {np.vdot(ket_1, ket_1)}")  # Should be 1
print(f"⟨0|1⟩ = {np.vdot(ket_0, ket_1)}")  # Should be 0 (orthogonal)
print(f"⟨1|0⟩ = {np.vdot(ket_1, ket_0)}")  # Should be 0

# Outer product (ket-bra)
print("\n=== Outer Products (Projection Operators) ===")
P0 = np.outer(ket_0, ket_0)  # |0⟩⟨0|
P1 = np.outer(ket_1, ket_1)  # |1⟩⟨1|
print(f"|0⟩⟨0| = \n{P0}")
print(f"\n|1⟩⟨1| = \n{P1}")

# Verify completeness relation
print("\n=== Completeness Relation ===")
print(f"|0⟩⟨0| + |1⟩⟨1| = \n{P0 + P1}")
print("This is the identity operator I - any state can be decomposed!")

### 2.2 Inner Product and Norms

The inner product in a quantum vector space satisfies:

1. **Linearity**: ⟨ψ|(α|φ⟩ + β|χ⟩) = α⟨ψ|φ⟩ + β⟨ψ|χ⟩
2. **Conjugate symmetry**: ⟨φ|ψ⟩ = ⟨ψ|φ⟩*
3. **Positive definiteness**: ⟨ψ|ψ⟩ > 0 for |ψ⟩ ≠ 0

The **norm** (or length) of a state is √⟨ψ|ψ⟩. For a valid quantum state, this must equal 1 (normalized states).

In [ ]:
# Explore inner products and norms

# Create some example states
psi = np.array([1, 1], dtype=complex) / np.sqrt(2)  # (|0⟩ + |1⟩)/√2
phi = np.array([1, -1], dtype=complex) / np.sqrt(2)  # (|0⟩ - |1⟩)/√2
chi = np.array([1, 1j], dtype=complex) / np.sqrt(2)  # (|0⟩ + i|1⟩)/√2

print("=== State Vectors ===")
print(f"|ψ⟩ = (|0⟩ + |1⟩)/√2: {psi}")
print(f"|φ⟩ = (|0⟩ - |1⟩)/√2: {phi}")
print(f"|χ⟩ = (|0⟩ + i|1⟩)/√2: {chi}")

print("\n=== Norms (should all be 1) ===")
print(f"||ψ⟩| = {np.sqrt(np.vdot(psi, psi)):.4f}")
print(f"||φ⟩| = {np.sqrt(np.vdot(phi, phi)):.4f}")
print(f"||χ⟩| = {np.sqrt(np.vdot(chi, chi)):.4f}")

print("\n=== Inner Products ===")
print(f"⟨ψ|φ⟩ = {np.vdot(psi, phi):.4f} (orthogonal!)")
print(f"⟨ψ|χ⟩ = {np.vdot(psi, chi):.4f}")
print(f"⟨φ|χ⟩ = {np.vdot(phi, chi):.4f}")

# Visualize on Bloch sphere
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_bloch_vector([1, 0, 0], title="|ψ⟩ = H|0⟩", ax=axes[0])
plot_bloch_vector([-1, 0, 0], title="|φ⟩ = H|1⟩", ax=axes[1])
plot_bloch_vector([0, 1, 0], title="|χ⟩", ax=axes[2])
plt.tight_layout()
plt.show()

### 2.3 Hilbert Space

A Hilbert space is a complete inner product space. For quantum computing, we work with finite-dimensional Hilbert spaces.

**Key properties:**
- Complete: All Cauchy sequences converge
- Inner product induces a norm
- Dimension determines the number of qubits

For n qubits, the Hilbert space has dimension 2ⁿ.

In [ ]:
# Explore Hilbert space dimensions

print("=== Hilbert Space Dimensions ===")
for n_qubits in range(1, 6):
    dim = 2 ** n_qubits
    print(f"{n_qubits} qubit(s): Dimension = {dim} (2^{n_qubits})")

print("\n=== Basis States for 2 Qubits ===")
# For 2 qubits, we have 4 basis states
basis_states = {
    '|00⟩': np.array([1, 0, 0, 0], dtype=complex),
    '|01⟩': np.array([0, 1, 0, 0], dtype=complex),
    '|10⟩': np.array([0, 0, 1, 0], dtype=complex),
    '|11⟩': np.array([0, 0, 0, 1], dtype=complex)
}

for name, state in basis_states.items():
    print(f"{name} = {state}")

# Verify orthonormality
print("\n=== Orthonormality Check ===")
for name1, state1 in basis_states.items():
    for name2, state2 in basis_states.items():
        inner = np.vdot(state1, state2)
        if np.abs(inner) > 0:
            print(f"⟨{name1[1:3]}|{name2[1:3]}⟩ = {inner:.4f}")

## 3. Matrix Operations in Quantum Computing

### 3.1 Unitary Matrices

Quantum gates must be **unitary** matrices. A unitary matrix U satisfies:

$$U^\dagger U = UU^\dagger = I$$

where U† is the conjugate transpose (Hermitian adjoint) of U.

**Why unitary?**
- Preserves normalization: if |ψ⟩ is normalized, U|ψ⟩ is also normalized
- Represents reversible quantum operations
- Ensures probability conservation

In [ ]:
# Explore unitary matrices

# Define common quantum gates
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

def check_unitary(matrix, name):
    """Check if a matrix is unitary"""
    dagger = matrix.conj().T
    product = dagger @ matrix
    identity = np.eye(matrix.shape[0])
    is_unitary = np.allclose(product, identity)
    print(f"{name}: U†U = I? {is_unitary}")
    return is_unitary

print("=== Unitary Matrix Verification ===\n")
gates = [('X', X), ('Y', Y), ('Z', Z), ('H', H), ('S', S), ('T', T)]
for name, gate in gates:
    check_unitary(gate, name)

### 3.2 Hermitian Matrices

Hermitian (or self-adjoint) matrices satisfy:

$$A = A^\dagger$$

**Key properties:**
- Eigenvalues are always real
- Eigenvectors form an orthonormal basis
- Can be diagonalized by a unitary transformation

In quantum mechanics:
- Observables are represented by Hermitian operators
- Pauli matrices are Hermitian
- Measurement projectors are Hermitian

In [ ]:
# Explore Hermitian matrices

def check_hermitian(matrix, name):
    """Check if a matrix is Hermitian"""
    dagger = matrix.conj().T
    is_hermitian = np.allclose(matrix, dagger)
    print(f"{name}: A = A†? {is_hermitian}")
    return is_hermitian

print("=== Hermitian Matrix Verification ===\n")
for name, gate in gates:
    check_hermitian(gate, name)

# Let's also check eigenvalues are real
print("\n=== Eigenvalues (should be real for Hermitian) ===")
for name, gate in [('X', X), ('Y', Y), ('Z', Z)]:
    eigenvalues, eigenvectors = np.linalg.eig(gate)
    print(f"{name} eigenvalues: {eigenvalues.real}")

### 3.3 Normal Matrices

A normal matrix commutes with its conjugate transpose:

$$[A, A^\dagger] = AA^\dagger - A^\dagger A = 0$$

**Key theorem (Spectral Theorem):**
Any normal matrix can be diagonalized by a unitary transformation:

$$A = U^\dagger D U$$

where D is a diagonal matrix of eigenvalues.

This is crucial for quantum mechanics because:
- Unitary matrices are normal
- Hermitian matrices are normal
- This guarantees diagonalizability

In [ ]:
# Explore normal matrices

# Create a non-Hermitian but normal matrix
# Example: a unitary that's not Hermitian
print("=== Normal Matrix Example ===")

# Phase gate S is unitary but not Hermitian
print(f"S matrix:\n{S}")
print(f"S† matrix:\n{S.conj().T}")

# Check normality
commutator = S @ S.conj().T - S.conj().T @ S
print(f"\n[S, S†] = \n{commutator}")
print(f"Is S normal? {np.allclose(commutator, 0)}")

# Create a non-normal matrix
non_normal = np.array([[1, 1], [0, 1]], dtype=complex)
print(f"\n=== Non-Normal Matrix ===")
print(f"Non-normal matrix:\n{non_normal}")
commutator_nn = non_normal @ non_normal.conj().T - non_normal.conj().T @ non_normal
print(f"[A, A†] = \n{commutator_nn}")
print(f"Is it normal? {np.allclose(commutator_nn, 0)}")

## 4. Eigenvalues and Eigenvectors

### 4.1 Spectral Decomposition

For any Hermitian matrix A, we can write:

$$A = \sum_i \lambda_i |\psi_i\rangle\langle\psi_i|$$

where λi are the real eigenvalues and |ψi⟩ are the corresponding eigenvectors.

This is called the **spectral decomposition** and is fundamental to quantum mechanics.

In [ ]:
# Explore spectral decomposition

print("=== Spectral Decomposition of Pauli Matrices ===\n")

for name, matrix in [('X', X), ('Y', Y), ('Z', Z)]:
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    print(f"--- {name} Gate ---")
    print(f"Eigenvalues: {eigenvalues.real}")
    print(f"Eigenvectors (columns):")
    for i, (ev, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
        print(f"  λ={ev.real:.0f}: |{name}⟩ = {vec.real}")
    print()

# Verify spectral decomposition for Z
print("=== Verify Spectral Decomposition for Z ===")
eigenvalues, eigenvectors = np.linalg.eig(Z)

# Reconstruct from spectral decomposition
P_plus = np.outer(eigenvectors[:, 0], eigenvectors[:, 0].conj())
P_minus = np.outer(eigenvectors[:, 1], eigenvectors[:, 1].conj())

reconstructed = eigenvalues[0] * P_plus + eigenvalues[1] * P_minus
print(f"Original Z:\n{Z}")
print(f"\nReconstructed:\n{reconstructed}")
print(f"Match? {np.allclose(Z, reconstructed)}")

### 4.2 Expectation Values

In quantum mechanics, the expectation value of an observable A in state |ψ⟩ is:

$$\langle A \rangle = \langle\psi|A|\psi\rangle$$

This can be computed using the spectral decomposition:

$$= \sum_i \lambda_i |\langle\psi_i|\psi\rangle|^2$$

The squared overlap |⟨ψi|ψ⟩|² is the probability of measuring eigenvalue λi.

In [ ]:
# Compute expectation values

# Define some states
states = {
    '|0⟩': np.array([1, 0], dtype=complex),
    '|1⟩': np.array([0, 1], dtype=complex),
    '|+⟩': np.array([1, 1], dtype=complex) / np.sqrt(2),
    '|-⟩': np.array([1, -1], dtype=complex) / np.sqrt(2),
    '|+i⟩': np.array([1, 1j], dtype=complex) / np.sqrt(2),
    '|-i⟩': np.array([1, -1j], dtype=complex) / np.sqrt(2)
}

def expectation_value(state, observable):
    """Compute ⟨ψ|A|ψ⟩"""
    return np.vdot(state, observable @ state).real

print("=== Expectation Values ===\n")
print("⟨X⟩ in different states:")
for name, state in states.items():
    exp_x = expectation_value(state, X)
    print(f"  {name}: ⟨X⟩ = {exp_x:.4f}")

print("\n⟨Z⟩ in different states:")
for name, state in states.items():
    exp_z = expectation_value(state, Z)
    print(f"  {name}: ⟨Z⟩ = {exp_z:.4f}")

print("\nNote: |+⟩ and |-⟩ have ⟨Z⟩ = 0 (equal superposition)")
print("Note: |0⟩ and |1⟩ have ⟨X⟩ = 0 (no superposition)")

### 4.3 Variance and Uncertainty

The uncertainty (standard deviation) of an observable A in state |ψ⟩ is:

$$\sigma_A = \sqrt{\langle A^2 \rangle - \langle A \rangle^2}$$

The **Heisenberg uncertainty principle** states that for two observables A and B:

$$\sigma_A \cdot \sigma_B \geq \frac{1}{2} |\langle[A, B]\rangle|$$

This is why we cannot simultaneously know position and momentum precisely!

In [ ]:
# Explore uncertainty relations

def variance(state, observable):
    """Compute variance of observable"""
    exp_A = expectation_value(state, observable)
    exp_A2 = expectation_value(state, observable @ observable)
    return exp_A2 - exp_A**2

def commutator(A, B):
    """Compute commutator [A, B]"""
    return A @ B - B @ A

print("=== Uncertainty Relations ===\n")

# Compute uncertainties for |+⟩ state
state = states['|+⟩']
sigma_x = np.sqrt(variance(state, X))
sigma_z = np.sqrt(variance(state, Z))

print(f"State |+⟩ (equal superposition in X basis):")
print(f"  σ(X) = {sigma_x:.4f}")
print(f"  σ(Z) = {sigma_z:.4f}")
print(f"  σ(X)·σ(Z) = {sigma_x * sigma_z:.4f}")

# Check the commutator [X, Z]
xz_comm = commutator(X, Z)
print(f"\n[X, Z] = \n{xz_comm}")

# The theoretical minimum uncertainty is 1/2
print(f"\nHeisenberg bound: σ(X)·σ(Z) ≥ 1/2 = 0.5")
print(f"Actual product: {sigma_x * sigma_z:.4f} (minimum uncertainty state!)")

## 5. Tensor Products

### 5.1 Building Multi-Qubit Systems

The tensor product (⊗) combines Hilbert spaces to form larger systems. For two qubits:

$$|\psi\rangle \otimes |\phi\rangle = |\psi\rangle|\phi\rangle = |\psi\phi\rangle$$

In matrix representation, the tensor product of two matrices A (m×n) and B (p×q) is a (mp)×(nq) matrix:

$$A \otimes B = \begin{pmatrix} a_{11}B & a_{12}B & \\ a_{21}B & a_{22}B & \end{pmatrix}$$

In [ ]:
# Explore tensor products

# Use numpy's kron (Kronecker product) for tensor product
print("=== Tensor Products (Kronecker Products) ===\n")

# Single qubit states
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

# Tensor product of |0⟩ ⊗ |0⟩
state_00 = np.kron(ket0, ket0)
print(f"|0⟩ ⊗ |0⟩ = {state_00}")

# Tensor product of |0⟩ ⊗ |1⟩
state_01 = np.kron(ket0, ket1)
print(f"|0⟩ ⊗ |1⟩ = {state_01}")

# Tensor product of |1⟩ ⊗ |0⟩
state_10 = np.kron(ket1, ket0)
print(f"|1⟩ ⊗ |0⟩ = {state_10}")

# Tensor product of |1⟩ ⊗ |1⟩
state_11 = np.kron(ket1, ket1)
print(f"|1⟩ ⊗ |1⟩ = {state_11}")

print("\n=== Gate Tensor Products ===")

# X ⊗ I
I = np.eye(2, dtype=complex)
X_I = np.kron(X, I)
print(f"\nX ⊗ I (X on first qubit):")
print(X_I)

# I ⊗ X
I_X = np.kron(I, X)
print(f"\nI ⊗ X (X on second qubit):")
print(I_X)

### 5.2 Entangled States via Tensor Products

Entangled states cannot be written as tensor products of individual states. This is a key quantum resource.

**Separable states:**
$$|\psi\rangle = |\psi_1\rangle \otimes |\psi_2\rangle$$

**Entangled states:**
$$|\psi\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$
This Bell state cannot be factored!

In [ ]:
# Explore entanglement and separability

print("=== Separable vs Entangled States ===\n")

# Create a separable state |+⟩ ⊗ |0⟩
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
separable = np.kron(plus, ket0)
print(f"|+⟩ ⊗ |0⟩ (separable):")
print(separable)

# Create Bell state (entangled)
bell = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
print(f"\n(|00⟩ + |11⟩)/√2 (entangled):")
print(bell)

# Test if Bell state can be factored
# Try to find single-qubit states a and b such that a ⊗ b = Bell
print("\n=== Testing Factorability ===")
print("Can the Bell state be written as |a⟩ ⊗ |b⟩?")
print("Let's check by trying to find such a decomposition...")

# For a product state, the reduced density matrix should be pure
from qiskit.quantum_info import partial_trace

# Use Qiskit to compute reduced density matrix
from qiskit.quantum_info import DensityMatrix

bell_state = Statevector(bell)
print(f"\nBell state vector: {bell_state}")

# Compute reduced states
print("\nReduced density matrices:")
print("(If pure → separable, If mixed → entangled)")

In [ ]:
# Compute reduced density matrices using pure Python

def partial_trace(rho, sys_dim, trace_qubit):
    """Compute partial trace over one qubit"""
    # rho is 4x4 density matrix, trace_qubit=0 or 1
    d = sys_dim  # dimension of each subsystem
    result = np.zeros((d, d), dtype=complex)
    
    if trace_qubit == 0:
        # Trace over first qubit
        for i in range(d):
            for j in range(d):
                result[i, j] = rho[i*d + j, i*d + j] + rho[i*d + j + d, i*d + j + d]
    else:
        # Trace over second qubit
        for i in range(d):
            for j in range(d):
                result[i, j] = rho[i, j] + rho[i+d, j+d]
    
    return result

# Create density matrix for Bell state
bell_dm = np.outer(bell, bell.conj())
print("Bell state density matrix:")
print(np.round(bell_dm, 4))

# Compute reduced density matrices
rho_0 = partial_trace(bell_dm, 2, 0)  # Trace over first qubit
rho_1 = partial_trace(bell_dm, 2, 1)  # Trace over second qubit

print("\n=== Reduced Density Matrices ===")
print(f"ρ₀ (after tracing qubit 0):\n{rho_0}")
print(f"\nρ₁ (after tracing qubit 1):\n{rho_1}")

# Check purity (Tr(ρ²) should be 1 for pure, <1 for mixed)
purity_0 = np.trace(rho_0 @ rho_0).real
purity_1 = np.trace(rho_1 @ rho_1).real
print(f"\nPurity of ρ₀: {purity_0:.4f}")
print(f"Purity of ρ₁: {purity_1:.4f}")
print("\nPurity < 1 means the state is entangled!")
print("(For separable states, reduced states would be pure)")

## 6. Linear Transformations and Operators

### 6.1 Operator Representation

Quantum operations are linear transformations on the Hilbert space. In matrix representation:

$$|\psi'\rangle = U|\psi\rangle$$

where U is a unitary operator.

Operators can be represented in different bases, and understanding this is key to quantum computing.

In [ ]:
# Explore linear transformations

print("=== Quantum Gates as Linear Transformations ===\n")

# Apply gates to basis states
states = [('|0⟩', ket0), ('|1⟩', ket1), ('|+⟩', plus), ('|-⟩', np.array([1, -1], dtype=complex)/np.sqrt(2))]

gates = [('X', X), ('Y', Y), ('Z', Z), ('H', H)]

print("Gate actions on basis states:")
for gate_name, gate in gates:
    print(f"\n--- {gate_name} gate ---")
    for state_name, state in states:
        result = gate @ state
        # Find closest basis state
        overlaps = {
            '|0⟩': np.vdot(ket0, result),
            '|1⟩': np.vdot(ket1, result),
            '|+⟩': np.vdot(plus, result),
            '|-⟩': np.vdot(np.array([1, -1], dtype=complex)/np.sqrt(2), result)
        }
        closest = max(overlaps, key=lambda k: abs(overlaps[k]))
        print(f"  {gate_name}{state_name} → {closest} (amplitude: {np.round(result, 3)})")

### 6.2 Operator Composition

Quantum circuits are composed by multiplying matrices. The order matters!

$$U_{total} = U_n \cdot U_{n-1} \cdot ... \cdot U_1 \cdot U_0$$

In [ ]:
# Explore operator composition

print("=== Operator Composition ===\n")

# H-X-H should equal Z (up to global phase)
HXH = H @ X @ H
print("H · X · H =")
print(HXH)
print(f"\nZ matrix:")
print(Z)
print(f"\nAre they equal? {np.allclose(HXH, Z)}")

# X-H-X should equal -H (different global phase)
XHX = X @ H @ X
print("\n--- X · H · X ---")
print(XHX)
print(f"\nH matrix:")
print(H)
print(f"\nAre they equal? {np.allclose(XHX, H)}")
print("Note: Different global phase - not physically distinguishable!")

# Circuit: H-X-Z
circuit = Z @ X @ H
print("\n--- Circuit: Z·X·H ---")
print(circuit)

### 6.3 Matrix Exponentials

Many quantum gates are generated by exponentiating Pauli matrices:

$$e^{i\theta P} = \cos(\theta)I + i\sin(\theta)P$$

This is a consequence of the Taylor expansion and the fact that Pauli matrices square to identity.

In [ ]:
# Explore matrix exponentials

from scipy.linalg import expm

def matrix_exp(A, theta):
    """Compute exp(i*theta*A)"""
    return expm(1j * theta * A)

print("=== Matrix Exponentials ===\n")

# RX gate: e^(-i*theta*X/2)
theta = np.pi / 4
RX = matrix_exp(-X/2, theta)
print(f"e^(-iθX/2) for θ = π/4:")
print(RX)

# Compare with Qiskit's RZ
from qiskit.circuit import Parameter
from qiskit.circuit.library import RXGate, RZGate

print("\n=== Rotation Gates ===")
print("RX(π/2) gate matrix:")
rx_gate = RXGate(theta)
print(rx_gate.to_matrix())

print("\nRZ(π/2) gate matrix:")
rz_gate = RZGate(theta)
print(rz_gate.to_matrix())

## 7. Singular Value Decomposition

### 7.1 Classical SVD

The singular value decomposition (SVD) of a matrix M is:

$$M = U \Sigma V^\dagger$$

where:
- U and V are unitary matrices
- Σ is a diagonal matrix of singular values

SVD is fundamental to many quantum algorithms, especially in quantum machine learning.

In [ ]:
# Explore Singular Value Decomposition

from numpy.linalg import svd

print("=== Singular Value Decomposition ===\n")

# Create a non-unitary matrix (e.g., a measurement operator)
M = np.array([[1, 1], [0, 1]], dtype=complex)
print(f"Matrix M:\n{M}")

# Compute SVD
U, s, Vh = svd(M)
print(f"\nSingular values: {s}")
print(f"U matrix:\n{U}")
print(f"V† matrix:\n{Vh}")

# Reconstruct
Sigma = np.diag(s)
reconstructed = U @ Sigma @ Vh
print(f"\nReconstructed M:\n{reconstructed}")
print(f"Match? {np.allclose(M, reconstructed)}")

### 7.2 Quantum SVD and Low-Rank Approximation

In quantum computing, SVD is used for:

- **Quantum state preparation** - Efficiently preparing states from classical data
- **Quantum principal component analysis** - Finding principal components
- **Quantum kernel methods** - Computing kernel matrices

The connection between classical SVD and quantum information is an active research area.

In [ ]:
# Explore low-rank approximation

print("=== Low-Rank Approximation via SVD ===\n")

# Create a rank-1 matrix
a = np.array([[1], [2]], dtype=complex)
b = np.array([[1, 1]], dtype=complex)
rank1_matrix = a @ b
print(f"Rank-1 matrix:\n{rank1_matrix}")

# SVD
U, s, Vh = svd(rank1_matrix)
print(f"\nSingular values: {s}")
print(f"Number of non-zero singular values: {np.sum(s > 1e-10)}")

# Create a full-rank matrix
full_rank = np.array([[1, 2], [3, 4]], dtype=complex)
print(f"\n=== Full Rank Matrix ===")
print(f"Full rank matrix:\n{full_rank}")

U, s, Vh = svd(full_rank)
print(f"\nSingular values: {s}")

# Low-rank approximation (keep top k singular values)
def low_rank_approx(M, k):
    """Approximate matrix using k singular values"""
    U, s, Vh = svd(M)
    # Keep only top k
    U_k = U[:, :k]
    s_k = s[:k]
    Vh_k = Vh[:k, :]
    return U_k @ np.diag(s_k) @ Vh_k

print("\n=== Low-Rank Approximations ===")
for k in [1, 2]:
    approx = low_rank_approx(full_rank, k)
    error = np.linalg.norm(full_rank - approx, 'fro')
    print(f"Rank-{k} approximation:\n{approx}")
    print(f"Frobenius norm error: {error:.4f}\n")

## 8. Summary and Key Concepts

Let's review the linear algebra foundations we've explored:

In [ ]:
# Create summary visualization

print("="*60)
print("QUANTUM LINEAR ALGEBRA - KEY CONCEPTS SUMMARY")
print("="*60)

summary = """
1. VECTOR SPACES (ℂ²)
   - Qubits live in 2D complex vector spaces
   - Inner product: ⟨φ|ψ⟩ = Σ φ*ᵢψᵢ
   - Norm: ||ψ|| = √⟨ψ|ψ⟩ = 1 for valid quantum states

2. MATRIX OPERATIONS
   - Unitary: U†U = I (preserves normalization)
   - Hermitian: A = A† (real eigenvalues)
   - Normal: [A, A†] = 0 (spectral theorem applies)

3. EIGENVALUES & EIGENVECTORS
   - Spectral decomposition: A = Σ λᵢ|ψᵢ⟩⟨ψᵢ|
   - Expectation: ⟨A⟩ = ⟨ψ|A|ψ⟩
   - Uncertainty: σₓσᵤ ≥ ½|[X,Y]|

4. TENSOR PRODUCTS
   - Combines multi-qubit Hilbert spaces
   - Dimension grows exponentially: dim = 2ⁿ
   - Entangled states cannot be factored

5. LINEAR TRANSFORMATIONS
   - Quantum gates = unitary matrices
   - Composition = matrix multiplication
   - exp(iθP) = cos(θ)I + i sin(θ)P for Pauli P

6. SINGULAR VALUE DECOMPOSITION
   - M = UΣV† for any matrix
   - Key for quantum machine learning
   - Low-rank approximation via truncated SVD
"""
print(summary)

# Visualize key relationships
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

concepts = [
Vector Space ℂ²", "Foundation"),
Inner Product", "Structure"),
Unitary Operators", "Evolution"),
Hermitian Operators", "Observables"),
Tensor Products", "Multi-qubit"),
SVD", "Decomposition")
]

y_positions = np.linspace(0.9, 0.1, len(concepts))
for i, (concept, desc) in enumerate(concepts):
    ax.text(0.1, y_positions[i], f"{i+1}. {concept}", fontsize=14, fontweight='bold')
    ax.text(0.4, y_positions[i], f"← {desc}", fontsize=12, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.title("Quantum Linear Algebra Concepts Map", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Exercises

### Exercise 1: Verify Unitary Properties
Create a function that verifies whether a given matrix is unitary. Test it on various quantum gates.

### Exercise 2: Spectral Decomposition
Compute the spectral decomposition of the Hadamard matrix H. Verify that H = Σ λᵢ|ψᵢ⟩⟨ψᵢ|.

### Exercise 3: Entanglement Detection
Write a function that determines whether a 2-qubit state is entangled by computing the purity of reduced density matrices.

### Exercise 4: Gate Composition
Find the matrix representation of the circuit: H-X-Z-H. What state does it produce from |0⟩?

### Exercise 5: Matrix Exponentials
Compute e^(iπX/4) and verify that it equals the RX(π/2) gate up to a global phase.

### Exercise 6: SVD for Quantum States
Apply SVD to a non-unitary matrix and interpret the singular values in the context of quantum information.

## 10. Further Reading and Resources

- **Michael Nielsen & Isaac Chuang**: "Quantum Computation and Quantum Information" (Chapter 2)
- **Qiskit Textbook**: https://qiskit.org/textbook/
- **Linear Algebra for Quantum Computing**: Various tutorials online

---

## Next Steps

In the next notebook, we'll explore **Entanglement and Superposition** in more depth, focusing on the quantum information-theoretic aspects and their applications in quantum algorithms.

---
*This notebook is part of the Quantum Machine Learning series. For more resources, visit the repository README.*